In [ ]:
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

# Render saved figures inline when executed in a notebook kernel.
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('matplotlib', 'inline')

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)  # Python, NumPy, and TensorFlow seeds
tf.config.experimental.enable_op_determinism()

PROJECT_ROOT = Path.cwd().resolve()
MODEL_DIR = PROJECT_ROOT / 'models'
FIGURE_DIR = PROJECT_ROOT / 'report_assets'
(PROJECT_ROOT / 'results').mkdir(exist_ok=True)
for folder in (MODEL_DIR, FIGURE_DIR):
    folder.mkdir(exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / '.checkpoints' / 'cnn'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
(X_train_full, y_train_full), (X_test_raw, y_test) = mnist.load_data()

In [ ]:
# Split the 60,000 training images into 50,000 train + 10,000 validation.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_full, y_train_full,  # the data to split (images and their labels together)
    test_size=10000,            # how many go into the second part (validation)
    random_state=SEED,         # fixed seed -> same shuffle every time, for everyone
    stratify=y_train_full,     # keep the same share of each digit in both parts
)

In [ ]:
# Normalize to 0-1 using float32.
X_train = X_train_raw.astype('float32') / 255.0
X_val = X_val_raw.astype('float32') / 255.0
X_test = X_test_raw.astype('float32') / 255.0

In [ ]:
# CNN shape: add a channel dimension (28, 28, 1)
X_train_cnn = X_train.reshape(len(X_train), 28, 28, 1)
X_val_cnn   = X_val.reshape(len(X_val), 28, 28, 1)
X_test_cnn  = X_test.reshape(len(X_test), 28, 28, 1)

In [ ]:
print("Train:", X_train_cnn.shape, "Val:", X_val_cnn.shape, "Test:", X_test_cnn.shape)
print("Pixel range:", X_train_cnn.min(), "to", X_train_cnn.max())

In [ ]:
assert X_train_cnn.shape == (50000, 28, 28, 1)
assert X_val_cnn.shape == (10000, 28, 28, 1)
assert X_test_cnn.shape == (10000, 28, 28, 1)
assert X_train_cnn.dtype == np.float32
assert 0.0 <= X_train_cnn.min() <= X_train_cnn.max() <= 1.0
assert set(np.unique(y_train)) == set(range(10))
print('Train:', X_train_cnn.shape, 'Validation:', X_val_cnn.shape, 'Test:', X_test_cnn.shape)
print('Training label counts:', np.bincount(y_train))

In [ ]:
MAX_EPOCHS = 20
PATIENCE = 3
EXPERIMENTS = [
    {'name': 'A', 'filters': [16, 32], 'learning_rate': 0.001, 'batch_size': 128, 'dense_units': 64},
    {'name': 'B', 'filters': [32, 64], 'learning_rate': 0.001, 'batch_size': 128, 'dense_units': 64},
    {'name': 'C', 'filters': [16, 32], 'learning_rate': 0.0003, 'batch_size': 128, 'dense_units': 64},
]

In [ ]:
def build_cnn(filters, learning_rate, dense_units):
    model = keras.Sequential([  # Sequential = layers stacked in order
        keras.Input(shape=(28, 28, 1)),  # input: 28x28 grayscale image
        layers.Conv2D(filters[0], (3, 3), activation='relu'),  # slide 3x3 filters, ReLU
        layers.MaxPooling2D((2, 2)),  # keep the max of each 2x2 block
        layers.Conv2D(filters[1], (3, 3), activation='relu'),  # stage 2
        layers.MaxPooling2D((2, 2)),  # keep the max of each 2x2 block
        layers.Flatten(),  # feature maps -> one long row
        layers.Dense(dense_units, activation='relu'),  # fully connected layer
        layers.Dense(10, activation='softmax'),  # one probability per digit
    ], name='mnist_cnn')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'],
    )
    return model

preview = build_cnn([16, 32], learning_rate=0.001, dense_units=64)
preview.summary()
del preview

In [ ]:
trial_results = []
histories = {}  # will hold the loss curves for each experiment

def run_cnn(config):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_cnn(config['filters'], config['learning_rate'], config['dense_units'])
    checkpoint_path = CHECKPOINT_DIR / f"cnn_{config['name']}.keras"
    # Prevent a checkpoint from an earlier run being mistaken for this trial.
    if checkpoint_path.exists():
        checkpoint_path.unlink()
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor='val_loss', mode='min', save_best_only=True,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss', mode='min', patience=PATIENCE,
            restore_best_weights=True,
        ),
        keras.callbacks.TerminateOnNaN(),
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train_cnn, y_train,
        validation_data=(X_val_cnn, y_val),
        epochs=MAX_EPOCHS, batch_size=config['batch_size'],
        callbacks=callbacks, verbose=2, shuffle=True,
    )
    train_seconds = time.perf_counter() - start
    values = history.history
    if not all(np.isfinite(values[key]).all() for key in ('loss', 'val_loss', 'accuracy', 'val_accuracy')):
        raise RuntimeError(f"Trial {config['name']} produced non-finite metrics; investigate before selecting a model.")
    if not checkpoint_path.exists():
        raise RuntimeError(f"No checkpoint saved for trial {config['name']}.")
    best_index = int(np.argmin(values['val_loss']))
    # Verify the metrics of the actual file that will be considered for deployment.
    saved_model = keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = saved_model.evaluate(X_val_cnn, y_val, verbose=0)
    row = {
        **config,
        'best_epoch': best_index + 1,
        'epochs_run': len(values['loss']),
        'val_loss': float(val_loss),
        'val_accuracy': float(val_accuracy),
        'train_accuracy_at_best_epoch': float(values['accuracy'][best_index]),
        'train_seconds': float(train_seconds),
        'parameters': int(saved_model.count_params()),
        'checkpoint': str(checkpoint_path.relative_to(PROJECT_ROOT)),
    }
    histories[config['name']] = values
    print(f"Trial {config['name']}: best epoch {row['best_epoch']}, validation accuracy {val_accuracy:.4%}, training {train_seconds:.1f}s")
    return row

In [ ]:
for config in EXPERIMENTS:
    trial_results.append(run_cnn(config))

results = pd.DataFrame(trial_results)
print(results[['name', 'filters', 'learning_rate', 'batch_size', 'dense_units',
               'best_epoch', 'epochs_run', 'val_accuracy', 'val_loss', 'train_seconds', 'parameters']].to_string(index=False))

#  plot the loss curves

In [ ]:
fig, axes = plt.subplots(len(EXPERIMENTS), 2, figsize=(12, 3 * len(EXPERIMENTS)))
for row, config in enumerate(EXPERIMENTS):
    h = histories[config['name']]
    epochs = np.arange(1, len(h['loss']) + 1)
    axes[row, 0].plot(epochs, h['loss'], label='Training')
    axes[row, 0].plot(epochs, h['val_loss'], label='Validation')
    axes[row, 0].set_title(f"Trial {config['name']} — loss")
    axes[row, 1].plot(epochs, h['accuracy'], label='Training')
    axes[row, 1].plot(epochs, h['val_accuracy'], label='Validation')
    axes[row, 1].set_title(f"Trial {config['name']} — accuracy")
    for ax in axes[row]:
        ax.set_xlabel('Epoch')
        ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'cnn_learning_curves.png', dpi=150)
plt.show()

# Save the selected model

In [ ]:
ranked = results.sort_values(
    ['val_accuracy', 'val_loss', 'parameters', 'name'],
    ascending=[False, True, True, True],
)
winner = ranked.iloc[0].to_dict()
FINAL_PATH = MODEL_DIR / 'cnn_final.keras'
# Copy the exact selected checkpoint; do not retrain on the test set.
shutil.copy2(PROJECT_ROOT / winner['checkpoint'], FINAL_PATH)
print('Chosen trial:', winner['name'])
print('Convolution filters:', winner['filters'], 'Dense units:', winner['dense_units'])
print('Learning rate:', winner['learning_rate'], 'Batch size:', winner['batch_size'])
print('Selected epoch:', winner['best_epoch'], 'Validation accuracy:', winner['val_accuracy'])
print('Saved model:', FINAL_PATH)

# Evaluate

In [ ]:
keras.backend.clear_session()
load_start = time.perf_counter()
cnn_loaded = keras.models.load_model(FINAL_PATH)
load_seconds = time.perf_counter() - load_start

# Warm up using validation data, leaving test evaluation to the next prediction.
_ = cnn_loaded.predict(X_val_cnn[:128], batch_size=128, verbose=0)
start = time.perf_counter()
test_probs = cnn_loaded.predict(X_test_cnn, batch_size=128, verbose=0)
inference_seconds = time.perf_counter() - start
assert test_probs.shape == (10000, 10)
assert np.isfinite(test_probs).all()
assert np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-5)
test_pred = np.argmax(test_probs, axis=1)  # the digit with the highest probability
test_accuracy = float(accuracy_score(y_test, test_pred))
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, test_pred, average='macro', zero_division=0,
)
# Predictive cross-entropy excludes any L2 weight penalty.
true_class_probs = test_probs[np.arange(len(y_test)), y_test]
test_crossentropy = float(-np.log(np.clip(true_class_probs, 1e-7, 1.0)).mean())
report = classification_report(y_test, test_pred, labels=list(range(10)), output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, test_pred, labels=list(range(10)))

print(f'Test accuracy: {test_accuracy:.4%}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Predictive cross-entropy: {test_crossentropy:.4f}')
print(f'Model size: {FINAL_PATH.stat().st_size / (1024 ** 2):.2f} MiB')
print(f'Loading: {load_seconds:.2f}s; 10,000 predictions after warm-up: {inference_seconds:.2f}s')
print(classification_report(y_test, test_pred, labels=list(range(10)), digits=4, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Selected CNN — test confusion matrix')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'cnn_confusion_matrix.png', dpi=150)
plt.show()

wrong = np.flatnonzero(test_pred != y_test)[:12]
if len(wrong):
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, wrong):
        ax.imshow(X_test_raw[index], cmap='gray')
        ax.set_title(f'True: {y_test[index]} / Pred: {test_pred[index]}')
    fig.tight_layout()
    plt.show()
else:
    print('No test errors in this run.')

# Verify reload and update model comparison

In [ ]:
second_load = keras.models.load_model(FINAL_PATH, compile=False)
validation_probs_1 = cnn_loaded.predict(X_val_cnn[:32], verbose=0)
validation_probs_2 = second_load.predict(X_val_cnn[:32], verbose=0)
np.testing.assert_allclose(validation_probs_1, validation_probs_2, rtol=1e-5, atol=1e-6)
assert second_load.input_shape == (None, 28, 28, 1)
assert second_load.output_shape == (None, 10)
del second_load
print('Reload check passed; input (N, 28, 28, 1), output (N, 10).')

In [ ]:
comparison_row = {
    'model': 'CNN',
    'selected_trial': str(winner['name']),
    'validation_accuracy': float(winner['val_accuracy']),
    'test_accuracy': test_accuracy,
    'macro_f1': float(macro_f1),
    'file_size_mib': FINAL_PATH.stat().st_size / (1024 ** 2),
}
comparison_path = PROJECT_ROOT / 'results' / 'comparison.csv'
previous = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if not previous.empty:
    previous = previous.loc[previous['model'] != comparison_row['model']]
comparison = pd.concat([previous, pd.DataFrame([comparison_row])], ignore_index=True)
comparison = comparison[list(comparison_row)].sort_values('model').reset_index(drop=True)
comparison.to_csv(comparison_path, index=False)
print(pd.DataFrame([comparison_row]).to_string(index=False))

# Recorded experiment results

These measurements describe the experiments used to select the included saved model. They are recorded results, not execution outputs from rerunning this notebook.

| Trial | Validation accuracy | Fit seconds |
| --- | --- | --- |
| A | 98.57% | 50.40 |
| B | 98.74% | 54.91 |
| C | 98.63% | 107.50 |

Selected trial: **B**. Test accuracy: **99.02%**; macro F1: **0.9901**.

[Test confusion matrix](report_assets/cnn_confusion_matrix.png). [Training and validation curves](report_assets/cnn_learning_curves.png).

See [the report](report.md) for the full configuration comparison and model recommendation. If you rerun training, update this recorded summary and the report with the new outputs.